# Phase 1: environment, baselines, and nominal IPPO
This notebook is a command dashboard. The implementation lives in `robosoccer/`; runs are copied to Drive only after metadata says they are complete. Pymunk evaluation is sim-to-sim transfer, not physical deployment.

> **Repaired benchmark:** rerun this notebook from the top. Do not combine pre-repair checkpoints or evaluation tables with the new results. The inexpensive baseline gate must pass before nominal IPPO training begins, and the final Phase-1 audit must pass before opening Phase 2.

## 1–2. Initialization variables

In [ ]:
from pathlib import Path
import json, os, shutil, sys
REPO_URL = "https://github.com/djdhillxn/football"
REPO_DIR = Path("/content/robot-soccer-transfer")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT = DRIVE_MOUNT / "MyDrive" / "RobotSoccerTransfer"
print(REPO_DIR, DRIVE_PROJECT)

## 3. Runtime and GPU check

In [ ]:
!python --version
!nvidia-smi || true
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())

## 4. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount(str(DRIVE_MOUNT))
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)

## 5. Clone or fast-forward the repository

In [ ]:
if REPO_DIR.exists():
    dirty = !git -C {REPO_DIR} status --porcelain
    if dirty:
        raise RuntimeError("Repository has local changes; commit or remove them before pulling.")
    !git -C {REPO_DIR} fetch origin --quiet
    !git -C {REPO_DIR} pull --ff-only
else:
    if REPO_URL.startswith("REPLACE_"):
        raise RuntimeError("Set REPO_URL before cloning.")
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git rev-parse HEAD

In [ ]:
%pwd

In [ ]:
%ls

## 6. Install and verify the package

In [ ]:
# Preserve Colab's CUDA-enabled Torch, NumPy, and pandas builds.
!python -m pip install -q gymnasium==1.2.1 pettingzoo==1.26.1 pymunk==7.3.0 PyYAML==6.0.3 tensorboard==2.20.0 imageio==2.37.2 imageio-ffmpeg==0.6.0 pytest==8.4.2 ruff==0.14.5
if int(get_ipython().user_ns.get("_exit_code", 0)) != 0:
    raise RuntimeError("dependency installation failed")
!python -m pip install -e . --no-deps -q
if int(get_ipython().user_ns.get("_exit_code", 0)) != 0:
    raise RuntimeError("editable package installation failed")
from importlib.metadata import version
for package in ["robosoccer-transfer", "torch", "numpy", "pandas", "gymnasium", "pettingzoo", "pymunk"]:
    print(package, version(package))
sys.path.insert(0, str(REPO_DIR))
import robosoccer
print("robosoccer", robosoccer.__version__)

## 7. Shared artifact helpers

In [ ]:
def require_shell_success(label):
    status = int(get_ipython().user_ns.get("_exit_code", 0))
    if status != 0:
        raise RuntimeError(f"{label} failed with shell status {status}")

def latest_run(name):
    pointer = REPO_DIR / "runs" / f"latest_{name}.txt"
    if not pointer.is_file():
        raise FileNotFoundError(f"Missing latest-run pointer: {pointer}")
    return Path(pointer.read_text().strip())

def display_json(path):
    data = json.loads(Path(path).read_text())
    print(json.dumps(data, indent=2)[:5000])
    return data

from robosoccer.config import load_config
from robosoccer.evaluation import phase1_readiness_audit

def run_phase1_audit(baseline_run, ippo_run=None):
    baseline_summary = json.loads((Path(baseline_run) / "eval" / "baseline_summary.json").read_text())
    abstract_summary = None
    transfer_summary = None
    if ippo_run is not None:
        abstract_path = Path(ippo_run) / "eval" / "abstract_standard" / "summary.json"
        transfer_path = Path(ippo_run) / "eval" / "pymunk_transfer" / "summary.json"
        abstract_summary = json.loads(abstract_path.read_text()) if abstract_path.is_file() else None
        transfer_summary = json.loads(transfer_path.read_text()) if transfer_path.is_file() else None
    audit = phase1_readiness_audit(
        load_config(REPO_DIR / "configs" / "base.yaml"),
        baseline_summary,
        abstract_summary,
        transfer_summary,
    )
    output_run = Path(ippo_run) if ippo_run is not None else Path(baseline_run)
    audit_path = output_run / "eval" / "phase1_readiness.json"
    audit_path.write_text(json.dumps(audit, indent=2) + "\n")
    print(json.dumps(audit, indent=2))
    return audit

def copy_completed_run(run_dir):
    run_dir = Path(run_dir)
    metadata = json.loads((run_dir / "run_metadata.json").read_text())
    if metadata.get("status") != "complete":
        raise RuntimeError(f"Refusing to sync incomplete run: {run_dir}")
    destination = DRIVE_PROJECT / "runs" / run_dir.name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(run_dir, destination, dirs_exist_ok=True)
    return destination

def restore_run(name):
    source = DRIVE_PROJECT / "runs" / name
    destination = REPO_DIR / "runs" / name
    if not source.is_dir():
        raise FileNotFoundError(source)
    shutil.copytree(source, destination, dirs_exist_ok=True)
    metadata = json.loads((destination / "run_metadata.json").read_text())
    pointer = REPO_DIR / "runs" / f"latest_{metadata['experiment_name']}.txt"
    pointer.parent.mkdir(parents=True, exist_ok=True)
    pointer.write_text(str(destination.resolve()) + "\n")
    return destination

def sync_reports_and_runs():
    shutil.copytree(REPO_DIR / "reports", DRIVE_PROJECT / "reports", dirs_exist_ok=True)
    for pointer in (REPO_DIR / "runs").glob("latest_*.txt"):
        shutil.copy2(pointer, DRIVE_PROJECT / pointer.name)

## 8. Run the full test suite

In [ ]:
!python -m pytest -q
require_shell_success("pytest")

## 9. Run the CPU smoke configuration

In [ ]:
!python -m scripts.train --config configs/smoke_test.yaml
require_shell_success("smoke training")
display_json(latest_run("smoke_test") / "run_metadata.json")

## 10. Run the PettingZoo API validation only

In [ ]:
!python -m pytest -q tests/test_core.py -k parallel_api
require_shell_success("PettingZoo parallel API tests")

## 11. Record a random-policy environment video

In [ ]:
!python -m scripts.record_video --config configs/base.yaml --baseline random --simulator abstract --episodes 1
require_shell_success("random-policy video")

## 12–13. Evaluate scripted baselines and enforce the pre-training gate
This paired-seed, 100-episode evaluation must show a non-saturated, policy-sensitive Pymunk task and useful role coordination before spending GPU time on IPPO.

In [ ]:
!python -m scripts.evaluate_baselines --config configs/base.yaml --episodes 100
require_shell_success("baseline evaluation")
baseline_run = latest_run("base")
display_json(baseline_run / "eval" / "baseline_summary.json")
baseline_audit = run_phase1_audit(baseline_run)
if not baseline_audit["baseline_ready"]:
    raise RuntimeError("Baseline gate failed. Do not begin nominal IPPO training.")

## 14. Train nominal IPPO
Run this only after the baseline gate above passes. This creates a fresh post-repair checkpoint; the older Phase-1 actor is retained only as historical evidence.

In [ ]:
!python -m scripts.train --config configs/ippo_nominal.yaml
require_shell_success("nominal IPPO training")

## 15. Evaluate IPPO in the abstract simulator

In [ ]:
ippo_run = latest_run("ippo_nominal")
!python -m scripts.evaluate --run-dir {ippo_run} --checkpoint best --simulator abstract --suite standard
require_shell_success("IPPO abstract evaluation")

## 16. Evaluate the same frozen IPPO actor in Pymunk

In [ ]:
ippo_run = latest_run("ippo_nominal")
!python -m scripts.evaluate --run-dir {ippo_run} --checkpoint best --simulator pymunk --suite transfer
require_shell_success("IPPO Pymunk transfer evaluation")
phase1_audit = run_phase1_audit(baseline_run, ippo_run)
if phase1_audit["phase2_ready"]:
    print("PHASE 2 GO: every Phase-1 safeguard passed.")
else:
    print("PHASE 2 NO-GO: inspect phase1_readiness.json before continuing.")

## 17. Record IPPO videos in both simulators

In [ ]:
ippo_run = latest_run("ippo_nominal")
!python -m scripts.record_video --run-dir {ippo_run} --checkpoint best --simulator abstract --episodes 3
require_shell_success("IPPO abstract videos")
!python -m scripts.record_video --run-dir {ippo_run} --checkpoint best --simulator pymunk --episodes 3
require_shell_success("IPPO Pymunk videos")

## 18. Sync completed artifacts to Drive

In [ ]:
for name in ["smoke_test", "ippo_nominal"]:
    print(copy_completed_run(latest_run(name)))
print(copy_completed_run(baseline_run))
sync_reports_and_runs()

## 19. Finished
Verify the Drive copies and `eval/phase1_readiness.json`, then disconnect and delete the Colab runtime. Start Phase 2 only when `phase2_ready` is `true`; full runs are expensive, so do not leave an idle accelerator attached.